# ST Score Restore — Stage 11 V2d Crash-Safe Colab Recovery

Development-only, inference-only recovery runner. Restore remains exploratory GPU work; Oemer detector inference is pinned to ONNX Runtime 1.20.1 CPUExecutionProvider. All reusable exact inputs, source/restored pages, and detector-page progress persist in Google Drive.


In [ ]:
# 1) Pinned detector runtime. CUDA is checked later only if Restore cache is incomplete.
!nvidia-smi || true
!apt-get -qq update
!apt-get -qq install -y poppler-utils
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1 || true
!pip -q install "onnxruntime==1.20.1" scipy scikit-learn scikit-image matplotlib pillow typing-extensions
!pip -q install --no-deps "git+https://github.com/BreezeWhite/oemer@dbe2a933d630d0f74805d717960eb259473f5978"

import sys, subprocess
print('python', sys.version)
if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError('This repository supports Python 3.11-3.12; Python 3.13 is not claimed supported.')
subprocess.run([sys.executable, '-c', "import onnxruntime as o; print('detector ORT', o.__version__, o.get_available_providers()); assert o.__version__ == '1.20.1'; assert 'CPUExecutionProvider' in o.get_available_providers()"], check=True)


In [ ]:
# 2) Mount Drive and refresh the exact PR branch source.
from google.colab import drive
drive.mount('/content/drive')

import shutil, subprocess, sys
from pathlib import Path
REPO = Path('/content/st-score-restore-engine')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '-q', '-b', 'stage11-v2c-semantic-detector-corpus-expansion', 'https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
print('repo ready', REPO)


In [ ]:
# 3) Always run the robust recovery runner in a fresh subprocess and tee all output to Drive.
import os, subprocess, sys
from pathlib import Path
RESULT_DIR = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_RESULTS')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
LOG = RESULT_DIR / 'v2d_recovery_full.log'
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONFAULTHANDLER'] = '1'
env['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + env.get('PYTHONPATH', '')
cmd = [sys.executable, '-m', 'st_score_restore.stage11_v2d_colab_runner', '--run']
print('running:', ' '.join(cmd))
print('log:', LOG)
with LOG.open('w', encoding='utf-8', buffering=1) as log_handle:
    proc = subprocess.Popen(
        cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        log_handle.write(line)
    returncode = proc.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, cmd)


## Başarı işaretleri

`Oemer preflight PASS` → `Oemer real-image smoke PASS: source + restored` → `V2D RECOVERY PASS` → `SAVED:`.

Aynı notebook yeniden çalıştırıldığında doğrulanmış Drive cache ve sayfa-bazlı detector JSON kayıtları yeniden kullanılır. Final canonical CPU rerun gereksinimi değişmez.
